In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
import sys
from pathlib import Path
from openai import OpenAI


import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve().parent
sys.path.append(str(PROJECT_ROOT))
from WebScrapper.scrape_site import scrape_site


In [2]:
load_dotenv(override=True)
api_key = os.getenv("GEMMA_API_KEY")

if api_key and len(api_key) > 10:
    print("API Key loaded successfully.")
else:
    print("Failed to load API Key. Please check your .env file.")

MODEL = "gemma3:latest"
GEMMA_BASE_URL = os.getenv("GEMMA_BASE_URL")

gemma = OpenAI(base_url=GEMMA_BASE_URL, api_key=api_key)

API Key loaded successfully.


In [3]:
link_system_prompt = """
You are provided with a list of website links and the contents of those web pages. 
You are able to decide which links are relevant and must include in the creative brochure, such as About page or Company Page etc.

You should respond in JSON as in this example:

{
  "start_url": "https://www.mcdonalds.com/ca/en-ca.html",
  "pages_scraped": 15,
  "pages": [
    {
      "title": "Title of the page",
      "text": "Contents of the page",
      "url": "https://url.com"
    },

"""

In [4]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links and contents on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email and Discord links.

Links (some might be relative links):

"""
    links = scrape_site(url)
    #print(links)
    user_prompt += "\n".join(links)
    return user_prompt

In [5]:
def select_relevant_links(url):
    response = gemma.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )

    result = response.choices[0].message.content
    links = json.loads(result)
    return links

In [56]:
select_relevant_links("https://www.apple.com")

{'start_url': 'https://www.apple.com/',
 'pages_scraped': 35,
 'pages': [{'title': 'Apple - Our Products',
   'text': "Discover Apple's innovative products, including iPhone, iPad, Mac, Apple Watch, AirPods, and Apple TV. Experience the seamless integration of hardware and software powered by the Apple ecosystem.",
   'url': 'https://www.apple.com/ca/shop/products'},
  {'title': 'Apple - Our Services',
   'text': 'Explore Apple Services, including Apple Music, Apple TV+, Apple Arcade, iCloud, AppleCare, and more. Enhance your digital life with a suite of services designed for seamless connectivity and entertainment.',
   'url': 'https://www.apple.com/ca/shop/services'},
  {'title': 'Apple - About Apple',
   'text': 'Learn about Apple’s history, mission, values, and commitment to innovation. Discover how Apple creates products and services that empower people around the world.',
   'url': 'https://www.apple.com/ca/about-apple/'},
  {'title': 'Apple - Newsroom',
   'text': 'Stay up-to-da

In [9]:
brochure_system_prompt = """
You are a sales expert and copywriter. Your task is to create a compelling and engaging brochure for a company based on the provided web page contents.
you always generate creative and catchy brochure content that atracts customers. Creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [10]:
def format_scraped_content(scraped_data: dict) -> str:
    output = []

    for page in scraped_data.get("pages", []):
        title = page.get("title", "")
        url = page.get("url", "")
        text = page.get("text", "")

        output.append(f"## {title}\nURL: {url}\n{text}\n")

    return "\n".join(output)


In [11]:
def get_brochure_user_prompt(company_name, url):
    scraped_data = select_relevant_links(url)  # returns dict
    content_text = format_scraped_content(scraped_data)  # convert to str

    user_prompt = f"""You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages.
Use this information to build a short brochure of the company in markdown without code blocks.

{content_text}
"""
    return user_prompt


In [12]:
get_brochure_user_prompt("Apple", "https://www.apple.com")

'You are looking at a company called: Apple\nHere are the contents of its landing page and other relevant pages.\nUse this information to build a short brochure of the company in markdown without code blocks.\n\n\n'

In [13]:
def create_brochure(company_name, url):
    response = gemma.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [14]:
create_brochure("Striver-DSA Course", "https://takeuforward.org/dsa/strivers-a2z-sheet-learn-dsa-a-to-z")

## Unlock Your Potential with Striver-DSA Course

**(Image: A dynamic graphic depicting coding, problem-solving, and growth – perhaps a stylized brain connecting to code.)**

**Are you ready to conquer the challenges of Data Structures and Algorithms?** Striver-DSA Course provides a comprehensive, step-by-step approach to mastering the core concepts that underpin computer science. Whether you’re a budding developer, a student preparing for technical interviews, or simply curious about the world of algorithms, we’ve got you covered.

**What You’ll Learn:**

Our A-to-Z course breaks down complex topics into manageable modules, covering everything from foundational data structures to advanced algorithmic techniques. You’ll gain practical knowledge and problem-solving skills through:

*   **Comprehensive Coverage:** From Arrays and Linked Lists to Graphs, Dynamic Programming, and more – we tackle them all!
*   **Clear Explanations:** Our intuitive explanations make even the most challenging concepts easy to understand.
*   **Hands-on Practice:** Implement algorithms and solve coding problems to solidify your learning.
*   **Diverse Algorithms:**  Learn about Sorting Algorithms (Bubble Sort, Merge Sort, Quick Sort), Searching Algorithms (Linear Search, Binary Search) and many others.



**For Whom is Striver-DSA Course?**

*   **Aspiring Developers:** Build a solid foundation for a successful career in software development.
*   **Students:** Prepare for technical interviews and excel in computer science courses.
*   **Career Shifters:**  Gain the in-demand skills to transition into a rewarding tech role.
*   **Anyone Curious:**  Explore the fascinating world of algorithms and problem-solving.



**Join Our Community - Invest in Yourself**

**(Small Image: Diverse group of people collaborating on a coding project)**

**For Investors:**

Striver-DSA Course is rapidly becoming the go-to platform for aspiring computer scientists. We’re seeing strong growth in enrollments and a highly engaged student base, driven by a dedicated team committed to providing exceptional learning resources.  Invest in our mission to empower the next generation of problem-solvers!

**Careers with Striver:**

**(Image:  A person confidently working at a computer with a skyline in the background)**

**We’re Hiring!**  Join our growing team of passionate educators and developers.  We’re looking for individuals who:

*   Are passionate about making quality DSA learning accessible to all.
*   Have a strong understanding of computer science fundamentals.
*   Are collaborative, driven, and eager to make an impact.

**Currently Seeking:**  Software Engineers, Content Developers, and Community Managers.  Visit [Link to Careers Page – Would need to be added] to explore opportunities.

**Start Your Journey Today!**

**Visit our website: [https://takeuforward.org/](https://takeuforward.org/)**

**(Small logo: Striver-DSA Course)**

**#DSA #Algorithms #Coding #Programming #DataStructures #TakeUForward**